# 实验2：逻辑回归（Softmax分类）—— MNIST手写数字识别

本实验使用 TensorFlow / Keras 构建一个最简单的 Softmax 逻辑回归模型，对 MNIST 手写数字进行分类。

**模型结构**：Input(784) → Dense(10) → Softmax

**学习目标**：
- 理解 Softmax 分类的原理
- 掌握 Keras Functional API 的使用
- 理解交叉熵损失函数
- 可视化模型权重，理解线性分类器的工作方式

---

### 核心理论背景

**什么是线性分类？** 线性分类器对每个类别学习一组权重 $\mathbf{w}_k$ 和偏置 $b_k$，计算输入 $\mathbf{x}$ 属于第 $k$ 类的得分：$z_k = \mathbf{w}_k^T \mathbf{x} + b_k$。直觉上，每组权重就是该类别的"模板"，得分越高说明输入与该模板越匹配。

**为什么需要 Softmax？** Dense 层输出的原始得分（logits）可能是任意实数，没有概率含义。Softmax 函数将这些得分转换为概率分布：

$$P(k) = \frac{e^{z_k}}{\sum_{j=0}^{9} e^{z_j}}$$

转换后所有输出都在 (0, 1) 之间，且总和为 1，可以解释为"属于每个类别的概率"。

**交叉熵损失函数**：衡量模型预测的概率分布与真实标签之间的差距。公式为 $L = -\sum_k y_k \log(\hat{y}_k)$。当真实标签是 One-Hot 编码时（例如数字3对应 $[0,0,0,1,0,...,0]$），交叉熵简化为 $L = -\log(\hat{y}_3)$。这意味着：模型对正确类别给出的概率越高，损失越小；如果模型"很自信但预测错误"（正确类别概率极低），损失会非常大。这种特性使交叉熵特别适合分类任务。

## 1. 导入库

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

# [旧写法] from keras.layers import Activation, Dense, Input
# [旧写法] from keras import Model
# [旧写法] from keras.optimizers import Adam
# [旧写法] from keras.utils import to_categorical
# ↑ 独立 keras 包在 TF 2.0+ 中已不推荐使用

# [新写法] 适用于 TensorFlow >= 2.0
from tensorflow.keras.layers import Activation, Dense, Input
from tensorflow.keras import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical

print('库导入完成')

## 2. 加载 MNIST 数据集

**MNIST 数据集简介**：MNIST 是深度学习领域最经典的入门数据集，被称为机器学习的"Hello World"。

- **内容**：手写数字图片（0~9），共10个类别
- **训练集**：60,000 张图片
- **测试集**：10,000 张图片
- **图片尺寸**：28 × 28 像素，灰度图（单通道），像素值范围 0~255（0=黑色，255=白色）
- **标签**：每张图片对应一个整数标签（0, 1, 2, ..., 9）

`mnist.load_data()` 会自动下载数据并返回训练集和测试集。返回的 `X_train` 形状为 `(60000, 28, 28)`，`Y_train` 形状为 `(60000,)`。

In [ ]:
# [旧写法] from tensorflow.examples.tutorials.mnist import input_data
# [旧写法] data = input_data.read_data_sets("data/MNIST/", one_hot=False)
# ↑ TF 1.x 的教程接口已废弃

# [新写法] 适用于 TensorFlow >= 2.0
(X_train, Y_train), (X_test, Y_test) = mnist.load_data()

print('训练集形状:', X_train.shape, Y_train.shape)
print('测试集形状:', X_test.shape, Y_test.shape)
print('像素值范围:', X_train.min(), '~', X_train.max())

## 3. 可视化样本

**为什么要可视化？** 在开始建模之前，先直观地看看数据长什么样是一个好习惯。可视化可以帮助我们：
1. **确认数据加载正确**：图片和标签是否对应、是否有损坏的数据；
2. **理解任务难度**：有些手写数字非常潦草，连人类也难以辨认（比如某些 4 和 9 很相似），这帮助我们对模型的预期准确率有合理的预期；
3. **发现数据问题**：比如是否有类别不均衡、是否有异常样本等。

下面随机展示16张训练集中的手写数字图片及其标签：

In [ ]:
# 随机展示16张手写数字图片
fig, axes = plt.subplots(2, 8, figsize=(12, 3))
axes = axes.flatten()
indices = np.random.choice(len(X_train), 16, replace=False)
for i, idx in enumerate(indices):
    axes[i].imshow(X_train[idx], cmap='gray')
    axes[i].set_title(str(Y_train[idx]))
    axes[i].set_xticks([])
    axes[i].set_yticks([])
plt.suptitle('MNIST 样本示例')
plt.tight_layout()
plt.show()

## 4. 数据预处理 & One-Hot 编码

在将数据送入模型之前，需要进行三步关键的预处理：

**第一步：展平（Reshape）**
- 原始图片形状为 `(28, 28)` 的二维矩阵，但我们的模型输入是一维向量
- `reshape(-1, 784)` 将每张图片从 28×28 的矩阵"拉平"为长度 784 ($28 \times 28 = 784$) 的一维向量
- 参数 `-1` 让 NumPy 自动计算样本数量

**第二步：归一化（Normalization）**
- 原始像素值为 0~255 的整数，数值较大会导致梯度不稳定
- 除以 255.0 后，像素值映射到 [0, 1] 范围，使训练更加稳定和高效
- `astype('float32')` 将数据类型从整数转为浮点数，这是神经网络计算的必要步骤

**第三步：One-Hot 编码**
- 原始标签是整数（如 3, 7, 0），但 Softmax 输出的是10维概率向量
- One-Hot 编码将整数标签转为与模型输出同形状的向量，例如：
  - 数字 0 → `[1, 0, 0, 0, 0, 0, 0, 0, 0, 0]`
  - 数字 3 → `[0, 0, 0, 1, 0, 0, 0, 0, 0, 0]`
  - 数字 9 → `[0, 0, 0, 0, 0, 0, 0, 0, 0, 1]`
- 这样才能与 Softmax 的输出逐元素比较，计算交叉熵损失

In [ ]:
# 展平 + 归一化
X_train = X_train.reshape(-1, 784).astype('float32') / 255.0
X_test = X_test.reshape(-1, 784).astype('float32') / 255.0

# One-Hot 编码
# [旧写法] from keras.utils import to_categorical
# [新写法] 适用于 TensorFlow >= 2.0（已在上方导入）
YY_train = to_categorical(Y_train)
YY_test = to_categorical(Y_test)

print('处理后训练集形状:', X_train.shape)
print('One-Hot 标签形状:', YY_train.shape)
print('示例标签 Y_train[0] =', Y_train[0], '-> One-Hot:', YY_train[0])

## 5. 构建 Softmax 分类模型

使用 **Keras Functional API** 构建一个最简单的逻辑回归模型。

### Keras Functional API 简介
Keras 提供两种建模方式：Sequential（顺序堆叠）和 Functional（函数式）。Functional API 通过显式定义数据流来构建模型，写法形如 `x = Layer()(x)`，更加灵活，是工程中推荐使用的方式。

### 模型结构详解
- **输入层 `Input((784,))`**：声明输入数据的形状（784维向量），不做任何计算
- **Dense 层 `Dense(10)`**：全连接层，执行线性变换 $\mathbf{z} = \mathbf{W}\mathbf{x} + \mathbf{b}$
  - 权重矩阵 $\mathbf{W}$ 形状为 $(784, 10)$，偏置向量 $\mathbf{b}$ 形状为 $(10,)$
  - 共 $784 \times 10 + 10 = 7850$ 个可学习参数
  - 输出10个原始得分（logits），分别对应数字 0~9
- **Softmax 激活 `Activation('softmax')`**：将10个原始得分转换为概率分布
  - 公式：$P(k) = \frac{e^{z_k}}{\sum_{j=0}^{9} e^{z_j}}$
  - 输出的10个值都在 (0,1) 之间，且总和为 1
  - 例如输出 `[0.01, 0.02, 0.01, 0.85, 0.01, 0.02, 0.03, 0.01, 0.02, 0.02]` 表示模型认为有 85% 概率是数字 3

In [ ]:
# [旧写法] from keras.layers import Activation, Dense, Input
# [旧写法] from keras import Model
# [新写法] 适用于 TensorFlow >= 2.0（已在上方导入）

input_layer = Input((784,))
x = input_layer
x = Dense(10)(x)                    # 10个输出（对应10个数字）
x = Activation('softmax')(x)        # Softmax 激活函数
output_layer = x

model = Model(input_layer, output_layer)
model.summary()

## 6. 编译模型

`model.compile()` 用于**配置训练过程**，指定三个关键要素。注意：编译只是"做好准备"，并不会开始训练。

- **优化器（optimizer）**：`Adam(learning_rate=0.01)` — Adam 是最常用的优化器，在标准梯度下降基础上加入了动量和自适应学习率机制。`learning_rate=0.01` 控制每次参数更新的步幅：太大会导致震荡不收敛，太小则学习极慢。
- **损失函数（loss）**：`categorical_crossentropy` — 分类任务的标准损失函数。它衡量模型输出的概率分布与真实 One-Hot 标签之间的差距。与回归任务使用的 MSE 不同，交叉熵损失与 Softmax 搭配时梯度形式简洁（$\hat{y}_k - y_k$），训练更加稳定高效。
- **评估指标（metrics）**：`accuracy` — 准确率，即预测正确的样本数占总样本数的比例。训练过程中会同时显示训练集准确率（`accuracy`）和验证集准确率（`val_accuracy`）。

In [ ]:
model.compile(
    # [旧写法] optimizer=Adam(0.01),
    # ↑ 参数 lr 在 TF 2.11+ 中已废弃
    # [新写法] 适用于 TensorFlow >= 2.11
    optimizer=Adam(learning_rate=0.01),
    loss='categorical_crossentropy',     # 交叉熵损失
    metrics=['accuracy']                 # 评估指标：准确率
)

print('模型编译完成')

## 7. 训练模型

`model.fit()` 正式开始训练。模型会反复遍历训练数据，每次执行：前向传播 → 计算损失 → 反向传播 → 更新参数。

### 关键参数说明

- **`epochs=10`**：训练轮数。1个 epoch = 模型完整"看过"所有训练样本一遍。10个 epoch 意味着全部60000张图片会被重复学习10次。对于逻辑回归这种简单模型，10轮通常就能收敛。
- **`batch_size=1000`**：小批量大小。不是一次性送入全部60000个样本，而是每次取1000个样本计算梯度并更新参数。每个 epoch 需要 $60000 / 1000 = 60$ 次参数更新（称为60个 step/iteration）。
- **`validation_data=(X_test, YY_test)`**：验证集。每个 epoch 结束后，模型会在验证集上"考试"（只做前向传播，不更新参数），帮助我们监控泛化性能。

### 训练输出解读
每个 epoch 结束后会打印：
- `loss` / `accuracy`：训练集上的损失和准确率
- `val_loss` / `val_accuracy`：验证集上的损失和准确率

**关注过拟合信号**：如果训练准确率持续上升而验证准确率停滞或下降，说明模型开始"死记硬背"训练数据，泛化能力变差。

In [ ]:
history = model.fit(
    X_train, YY_train,
    validation_data=(X_test, YY_test),
    batch_size=1000,
    epochs=10
)

## 8. 可视化权重

逻辑回归模型的权重可以直接可视化为 28×28 的图像，这是理解线性分类器工作原理的绝佳方式。

### 为什么权重可以可视化为图像？

逻辑回归的预测公式为：$z_k = \mathbf{w}_k^T \mathbf{x} + b_k$，即第 $k$ 类的得分是输入像素向量 $\mathbf{x}$ 与权重向量 $\mathbf{w}_k$ 的**内积（点积）**。内积本质上衡量的是两个向量的"相似度"——当输入图片与某个类别的权重模式越匹配，该类别的得分就越高。

因此，每个类别的权重向量 $\mathbf{w}_k$（784维）重塑为 28×28 后，就像是该类别的"标准模板"。

### 如何解读权重图像？

使用 `seismic`（红蓝）色图进行可视化：
- **红色区域（正权重）**：如果输入图片在这些位置有较高像素值（有笔画），会**增加**该类别的得分。可以理解为"这个数字通常在这些位置有笔画"。
- **蓝色区域（负权重）**：如果输入图片在这些位置有较高像素值，会**降低**该类别的得分。可以理解为"这个数字通常不应该在这些位置有笔画"。
- **白色区域（接近零）**：这些位置对该类别的判断影响不大。

通俗来说，线性分类器是通过**模板匹配**来识别数字的。这也解释了为什么逻辑回归的准确率只有约92%——它只能学习一个"平均模板"，无法处理书写风格差异很大的情况（比如不同人写的"2"可能差别很大）。后续实验中的多层神经网络和卷积网络能学习更抽象的特征，从而大幅提升准确率。

In [ ]:
weights = model.layers[1].get_weights()[0]  # shape: (784, 10)
print('权重矩阵形状:', weights.shape)

plt.figure()
fig, ax = plt.subplots(2, 5, figsize=(12, 5))
ax = ax.flatten()
for i in range(10):
    Im = weights[:, i].reshape((28, 28))
    ax[i].imshow(Im, cmap='seismic')
    ax[i].set_title('数字 {}'.format(i))
    ax[i].set_xticks([])
    ax[i].set_yticks([])
plt.suptitle('各类别对应的权重可视化（红色=正权重，蓝色=负权重）')
plt.tight_layout()
plt.show()

## 总结

### 本实验要点

1. **逻辑回归本质**：逻辑回归是一个没有隐藏层的单层神经网络，通过 Softmax 函数将输出转为概率分布。

2. **交叉熵损失**：分类任务使用交叉熵损失函数，而非 MSE。交叉熵能更好地衡量预测概率分布与真实分布之间的差异。

3. **权重可视化**：线性分类器的权重可以直接解释 —— 红色区域表示正贡献（像素越亮越可能属于该类别），蓝色区域表示负贡献。

4. **局限性**：逻辑回归是线性模型，准确率约 92%。后续实验中，加入隐藏层（多层感知机/卷积网络）可以显著提升性能。

### 新旧写法对照

| 功能 | 旧写法 | 新写法 |
|------|--------|--------|
| 导入层 | `from keras.layers import Dense` | `from tensorflow.keras.layers import Dense` |
| 导入模型 | `from keras import Model` | `from tensorflow.keras import Model` |
| 导入优化器 | `from keras.optimizers import Adam` | `from tensorflow.keras.optimizers import Adam` |
| 学习率参数 | `Adam(lr=0.001)` | `Adam(learning_rate=0.001)` |
| One-Hot编码 | `from keras.utils import to_categorical` | `from tensorflow.keras.utils import to_categorical` |
| 数据集加载 | `tensorflow.examples.tutorials.mnist` | `tensorflow.keras.datasets.mnist` |